# Simple MNIST convnet

**Author:** [fchollet](https://twitter.com/fchollet)<br>
**Date created:** 2015/06/19<br>
**Last modified:** 2020/04/21<br>
**Description:** A simple convnet that achieves ~99% test accuracy on MNIST.

**REPRODUCIBILITY**

Full reproducibility of results requires setting seeds in several places since randomness comes from multiple sources. This cell must be added at the very top of the notebook.

In [8]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)   # Python hash randomness
random.seed(SEED)                           # Python built-in random
np.random.seed(SEED)                        # NumPy
tf.random.set_seed(SEED)                    # TensorFlow / Keras

Why each line is needed:

*   PYTHONHASHSEED Dict/set iteration order in Python internals
*   random.seedPython's own random module, used by some Keras utilities
*   np.random.seedNumPy ops (data shuffling, initialisers that fall back to numpy)
*   tf.random.set_seedWeight initialisation, dropout masks, data shuffling ops inside TF
*   shuffle=False Batch ordering during training — otherwise TF reshuffles each epoch independently

*   os.environ["TF_DETERMINISTIC_OPS"] = "1"   # forces deterministic GPU ops



*Full reproducibility is only guaranteed on the same machine, same GPU, same library versions. GPU floating-point operations are non-deterministic by default due to parallelism. If you need exact bit-for-bit reproducibility across machines, you additionally need:*

In [9]:
os.environ["TF_DETERMINISTIC_OPS"] = "1"   # forces deterministic GPU ops

## Setup

In [10]:
#import numpy as np
import keras
from keras import layers

## Prepare the data

In [11]:
# Model / data parameters
num_classes = 10
input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


## Build the model

In [12]:
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

## Train the model


**REPRODUCIBILITY**

In the training cell, add *shuffle=False* to model.fit()

In [13]:
batch_size = 128
epochs = 15

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

# model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs,
          validation_split=0.1, shuffle=False)   # <-- add this to prevent random shuffling

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8891 - loss: 0.3712 - val_accuracy: 0.9785 - val_loss: 0.0863
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9656 - loss: 0.1128 - val_accuracy: 0.9852 - val_loss: 0.0587
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9727 - loss: 0.0887 - val_accuracy: 0.9880 - val_loss: 0.0459
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9777 - loss: 0.0724 - val_accuracy: 0.9875 - val_loss: 0.0457
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9800 - loss: 0.0632 - val_accuracy: 0.9892 - val_loss: 0.0408
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9824 - loss: 0.0565 - val_accuracy: 0.9910 - val_loss: 0.0351
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9839 - loss: 0.0525 - val_accuracy: 0.9902 - val_loss: 0.0348
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9845 - loss: 0.0475 - val_accuracy: 0.

## Evaluate the trained model

*  Test loss: 0.0267093013972044
*  Test accuracy: 0.9915000200271606

In [14]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.0267093013972044
Test accuracy: 0.9915000200271606


## Relevant Chapters from Deep Learning with Python
- [Chapter 8: Image classification](https://deeplearningwithpython.io/chapters/chapter08_image-classification)
